# Preprocess data to csv

In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[1]))

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod

In [8]:
from utils import *

In [9]:
SEED = 42
LEARNING_RATE = 0.01
MAX_ITER = 10000

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# STEP 1: DATA INGESTION
file = 'Dry_Bean_Dataset.xlsx'
try:
    df = pd.read_excel(file)
except Exception as e:
    print(f"Error loading file: {e}")

# STEP 2: GLOBAL DATA CLEANING & IMPUTATION
# Identify numeric and categorical columns across the entire dataframe
numeric_cols = df.select_dtypes(include=[np.number]).columns
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns

# Fill missing numeric values with the median (Robust to outliers)
num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# Handle categorical features (excluding the 'Class' target column)
feature_non_numeric = [col for col in non_numeric_cols if col != 'Class']
if feature_non_numeric:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[feature_non_numeric] = cat_imputer.fit_transform(df[feature_non_numeric])
    
    # Convert text categories into numerical labels
    for col in feature_non_numeric:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

# STEP 3: EXPORTING CLEANED RAW DATA
# Saving the dataset after imputation but before normalization
df.to_csv('dataset.csv', index=False)
print("Data cleaned and saved to 'dataset.csv'.")

# STEP 4: FEATURE AND TARGET SEPARATION
X = df.drop('Class', axis=1)
y = df['Class']

# STEP 5: DATASET PARTITIONING (TRAIN-TEST SPLIT)
# Creating an 80/20 split to evaluate model performance later
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 6: TARGET VARIABLE ENCODING
# Converting bean names (Seki, etc.) into integer labels
target_le = LabelEncoder()
y_train = target_le.fit_transform(y_train)
y_test = target_le.transform(y_test)

# STEP 7: FEATURE SCALING (Z-SCORE NORMALIZATION)
# Centering features at 0 with unit variance to help model convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# STEP 8: FINAL PREPROCESSING VERIFICATION
print("--- Summary ---")
print(f"Total Features: {X_train_scaled.shape[1]}")
print(f"Encoded Classes: {target_le.classes_}")

Data cleaned and saved to 'dataset.csv'.
--- Summary ---
Total Features: 16
Encoded Classes: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']
